# Splunk SOAR container enrichment

1. List containers on a SOAR instance.
2. For each container, pull everything SOAR has on it: detail, artifacts, action runs, playbook runs, comments, notes, attachments, audit, workbook tasks/phases, approvals, evidence.
3. Work out the final disposition (status → status type, close time, custom fields).

Endpoints follow the [SOAR 8.7.0 REST reference](https://help.splunk.com/en/splunk-soar/soar-on-premises/rest-api-reference/8.7.0/using-the-splunk-soar-rest-api/query-for-data).

The `ph-auth-token` is read from `SOAR1_TOKEN` in the repo-root `.env` file (git-ignored; see `../.env.example`).

In [ ]:
import os

from dotenv import find_dotenv, load_dotenv
import json
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

SERVER = "https://prod-soar.mdr.tekstream.com"
load_dotenv(find_dotenv(usecwd=True))
TOKEN = os.environ.get("SOAR1_TOKEN", "")  # from .env

TEST_CONTAINER_ID = 226691  # used to try the enrichment on one container
MAX_CONTAINERS = 50         # cap for the listing while testing; None = all
CONTAINER_FILTERS = {}      # e.g. {"_filter_label": '"events"'}; status filters need the numeric id

# Only containers created in the last LOOKBACK_MONTHS, processed oldest -> newest
LOOKBACK_MONTHS = 6
SINCE = (pd.Timestamp.now(tz="UTC") - pd.DateOffset(months=LOOKBACK_MONTHS)).strftime("%Y-%m-%dT%H:%M:%SZ")
CONTAINER_FILTERS["_filter_create_time__gte"] = f'"{SINCE}"'

OUTPUT_DIR = Path("../data")

assert TOKEN, "set SOAR1_TOKEN in .env (see ../.env.example)"
HEADERS = {"ph-auth-token": TOKEN, "Content-Type": "application/json"}

## Helpers

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)


def get(path, params=None):
    """GET a SOAR REST path and return parsed JSON. Raises on HTTP errors."""
    r = session.get(f"{SERVER}{path}", params=params, allow_redirects=False, timeout=120)
    r.raise_for_status()
    return r.json()


def get_all(path, params=None, page_size=200, limit=None):
    """Page through a {count, num_pages, data} list endpoint and return all rows."""
    params = dict(params or {})
    rows, page = [], 0
    while True:
        body = get(path, {**params, "page": page, "page_size": page_size})
        if not isinstance(body, dict) or "data" not in body:
            return body  # not a paginated list endpoint
        rows.extend(body["data"])
        if limit and len(rows) >= limit:
            return rows[:limit]
        page += 1
        if page >= body.get("num_pages", 0):
            return rows


def safe(fn, *args, **kwargs):
    """Run fn; return its result, or an {"error": ...} dict so one bad endpoint doesn't stop the run."""
    try:
        return fn(*args, **kwargs)
    except requests.HTTPError as e:
        return {"error": f"{e.response.status_code} {e.response.text[:300]}"}
    except Exception as e:
        return {"error": repr(e)}


def show(obj, limit=5000):
    print(json.dumps(obj, indent=2, default=str)[:limit])

## Status map

A container is closed when its status belongs to the **resolved** status type, not only when it's literally named `closed` (SOAR allows custom statuses).

In [ ]:
statuses = get_all("/rest/container_status")
status_df = pd.DataFrame(statuses)
status_df

In [ ]:
# Look up status type by status id or name (the container field may hold either)
STATUS_TYPE = {}
for s in statuses:
    STATUS_TYPE[s.get("id")] = s.get("status_type")
    STATUS_TYPE[s.get("name")] = s.get("status_type")
STATUS_TYPE

## 1. List containers

In [ ]:
containers = get_all(
    "/rest/container",
    {"sort": "create_time", "order": "asc", **CONTAINER_FILTERS},
    limit=MAX_CONTAINERS,
)
print(f"{len(containers)} containers created since {SINCE}")

LIST_COLUMNS = [
    "id", "name", "label", "status", "severity", "sensitivity", "owner_id",
    "container_type", "create_time", "start_time", "close_time", "tags", "artifact_count",
]
containers_df = pd.DataFrame(containers)
containers_df["status_type"] = containers_df["status"].map(STATUS_TYPE)
containers_df[[c for c in LIST_COLUMNS + ["status_type"] if c in containers_df.columns]]

In [ ]:
# Save SOAR host + container ID for every listed container
SOAR_HOST = urlparse(SERVER).hostname

OUTPUT_DIR.mkdir(exist_ok=True)
containers_csv = OUTPUT_DIR / "containers.csv"
pd.DataFrame({
    "soar_host": SOAR_HOST,
    "container_id": [c["id"] for c in containers],
}).to_csv(containers_csv, index=False)
print(f"Wrote {len(containers)} rows to {containers_csv}")


## 2. Enrich one container

In [ ]:
def enrich(container_id):
    """Collect everything available for one container. Each section is independent."""
    base = f"/rest/container/{container_id}"
    detail = safe(get, base, "_annotation_authorized_users")

    audit_params = {"format": "json", "sort": "asc"}
    if isinstance(detail, dict) and detail.get("create_time"):
        audit_params["start"] = detail["create_time"]  # audit defaults to the last 30 days

    return {
        "container_id": container_id,
        "detail": detail,
        "artifacts": safe(get_all, f"{base}/artifacts", {"include_expensive": ""}),
        "actions": safe(get_all, f"{base}/actions"),
        "playbook_runs": safe(get_all, f"{base}/playbook_runs"),
        "comments": safe(get_all, f"{base}/comments", {"sort": "time", "order": "asc"}),
        "notes": safe(get_all, "/rest/note", {"_filter_container_id": container_id}),
        "attachments": safe(get_all, f"{base}/attachments"),
        "audit": safe(get, f"{base}/audit", audit_params),
        "workbook_tasks": safe(get_all, "/rest/workbook_task", {"_filter_container_id": container_id}),
        "workbook_phases": safe(get_all, "/rest/workflow_phase", {"_filter_container_id": container_id}),
        "approvals": safe(get, f"{base}/approvals"),
        "evidence": safe(get_all, "/rest/evidence", {"_special_content_type": "True", "_filter_container": container_id}),
    }


def disposition(enriched):
    """Pull the outcome-related fields out of an enriched container."""
    d = enriched["detail"] if isinstance(enriched["detail"], dict) else {}
    return {
        "container_id": enriched["container_id"],
        "status": d.get("status"),
        "status_type": STATUS_TYPE.get(d.get("status")),
        "closed": STATUS_TYPE.get(d.get("status")) == "resolved",
        "disposition": (d.get("custom_fields") or {}).get("Disposition"),
        "sa_decision": (d.get("custom_fields") or {}).get("SA Decision"),
        "create_time": d.get("create_time"),
        "open_time": d.get("open_time"),
        "close_time": d.get("close_time"),
        "owner": d.get("owner"),
        "closing_owner": d.get("closing_owner"),
        "severity": d.get("severity"),
        "label": d.get("label"),
        "custom_fields": d.get("custom_fields"),
        "tags": d.get("tags"),
    }


def summary(enriched):
    """Row counts (or error) per section."""
    out = {}
    for key, val in enriched.items():
        if key in ("container_id", "detail"):
            continue
        if isinstance(val, dict) and "error" in val:
            out[key] = val["error"][:80]
        elif isinstance(val, list):
            out[key] = len(val)
        elif isinstance(val, dict):
            out[key] = val.get("count", len(val.get("data", [])) if "data" in val else "dict")
        else:
            out[key] = type(val).__name__
    return out


test = enrich(TEST_CONTAINER_ID)
pd.Series(summary(test)).to_frame("rows / error")

In [ ]:
# Disposition candidates for the test container
show(disposition(test))

In [ ]:
# "Field Values" artifacts (the ones soar-2 pulls) and their CEF fields
artifacts = test["artifacts"] if isinstance(test["artifacts"], list) else []
pd.DataFrame([{"artifact_id": a.get("id"), "name": a.get("name"), **(a.get("cef") or {})} for a in artifacts])

In [ ]:
# Inspect any section in full, e.g. "detail", "actions", "playbook_runs", "comments", "audit"
show(test["detail"])

## 3. Enrich every listed container

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
out_path = OUTPUT_DIR / "enriched_containers.jsonl"

dispositions, summaries = [], []
with out_path.open("w") as f:
    for i, c in enumerate(containers, 1):
        enriched = enrich(c["id"])
        f.write(json.dumps(enriched, default=str) + "\n")
        dispositions.append(disposition(enriched))
        summaries.append({"container_id": c["id"], **summary(enriched)})
        print(f"\r{i}/{len(containers)} container {c['id']}", end="")

print(f"\nWrote {out_path}")

In [ ]:
disposition_df = pd.DataFrame(dispositions)
disposition_df

In [ ]:
summary_df = pd.DataFrame(summaries)
summary_df